## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [2]:
import os
from dotenv import load_dotenv
load_dotenv() ## aloading all the environment variable

groq_api_key=os.getenv("GROQ_API_KEY")




In [3]:
from langchain_groq import ChatGroq
model = ChatGroq(model="openai/gpt-oss-120b", groq_api_key=groq_api_key)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001C0827381A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001C082738C20>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi , My name is M Umar Alam and I am a Chief AI Engineer")])

AIMessage(content='Hello, M\u202fUmar\u202fAlam! It’s great to meet a Chief AI Engineer. How can I assist you today?', additional_kwargs={'reasoning_content': 'We need to respond. The user introduced themselves as a Chief AI Engineer. Probably they want a response, maybe ask how can I help. Follow policy: no disallowed content. Just respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 86, 'total_tokens': 163, 'completion_time': 0.161597557, 'completion_tokens_details': {'reasoning_tokens': 41}, 'prompt_time': 0.003500747, 'prompt_tokens_details': None, 'queue_time': 0.377284622, 'total_time': 0.165098304}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_5a93aea882', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--01a02a38-db4b-7052-b656-743ee122563d-0', usage_metadata={'input_tokens': 86, 'output_tokens': 77, 'total_tokens': 163})

In [5]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi , My name is M Umar Alam and I am a Chief AI Engineer"),
        AIMessage(content="Hello M Umar Alam! It's nice to meet you. \n\nAs a Chief AI Engineer, what kind of projects are you working on these days? \n\nI'm always eager to learn more about the exciting work being done in the field of AI.\n"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content='Your name is **M\u202fUmar\u202fAlam**, and you work as a **Chief AI Engineer**.', additional_kwargs={'reasoning_content': 'The user asks: "Hey What\'s my name and what do I do?" We have prior conversation where user introduced themselves as "M Umar Alam" and "Chief AI Engineer". So answer accordingly.'}, response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 154, 'total_tokens': 226, 'completion_time': 0.151274913, 'completion_tokens_details': {'reasoning_tokens': 40}, 'prompt_time': 0.006983171, 'prompt_tokens_details': None, 'queue_time': 0.534134188, 'total_time': 0.158258084}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e1a78f200e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--01a02a39-0470-78d0-b3c7-0e45fa1c1ab1-0', usage_metadata={'input_tokens': 154, 'output_tokens': 72, 'total_tokens': 226})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [5]:
!pip install langchain_community

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [7]:
config={"configurable":{"session_id":"chat1"}}

In [8]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is M Umar Alam and I am a Chief AI Engineer")],
    config=config
)

In [9]:
response.content

'Hello, M\u202fUmar\u202fAlam! It’s great to meet a Chief AI Engineer. How can I assist you today?'

In [10]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

AIMessage(content='Your name is **M\u202fUmar\u202fAlam**.', additional_kwargs={'reasoning_content': 'The user asks "What\'s my name?" The conversation: user introduced themselves as "M Umar Alam". So answer: your name is M Umar Alam. Should be concise.'}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 127, 'total_tokens': 184, 'completion_time': 0.117304939, 'completion_tokens_details': {'reasoning_tokens': 35}, 'prompt_time': 0.005245223, 'prompt_tokens_details': None, 'queue_time': 0.264899014, 'total_time': 0.122550162}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_b1dd3e7a63', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--01a02a39-c6b0-7862-9884-082cab8f29bd-0', usage_metadata={'input_tokens': 127, 'output_tokens': 57, 'total_tokens': 184})

In [11]:
## change the config-->session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'I’m not sure what name you’d like me to use—could you let me know?'

In [12]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config1
)
response.content

'Nice to meet you, John! How can I help you today?'

In [13]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'Your name is John.'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [14]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Amnswer all the question to the best of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [15]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is M Umar Alam")]})

AIMessage(content='Hello, M\u202fUmar\u202fAlam! Nice to meet you. How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "Hi My name is M Umar Alam". Likely they are greeting and introducing themselves. The assistant should respond politely, perhaps ask how can help. No disallowed content. So respond with greeting, ask how can assist.'}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 99, 'total_tokens': 179, 'completion_time': 0.166358109, 'completion_tokens_details': {'reasoning_tokens': 49}, 'prompt_time': 0.01423272, 'prompt_tokens_details': None, 'queue_time': 0.377452481, 'total_time': 0.180590829}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_6dedd2be22', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--01a02a3d-72c8-70d0-9cdd-640819d2e541-0', usage_metadata={'input_tokens': 99, 'output_tokens': 80, 'total_tokens': 179})

In [ ]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [17]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is M Umar Alam")],
    config=config
)

response

AIMessage(content='Hello, M\u202fUmar\u202fAlam! Nice to meet you. How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "Hi My name is M Umar Alam". Probably they are greeting and introducing themselves. We should respond politely, perhaps ask how we can help. No disallowed content. So respond friendly.'}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 99, 'total_tokens': 172, 'completion_time': 0.152799325, 'completion_tokens_details': {'reasoning_tokens': 42}, 'prompt_time': 0.00367135, 'prompt_tokens_details': None, 'queue_time': 0.26493041, 'total_time': 0.156470675}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_803c0ba83d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--01a02a3d-e6ff-7f10-b6e5-7bddea99f972-0', usage_metadata={'input_tokens': 99, 'output_tokens': 73, 'total_tokens': 172})

In [18]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is **M\u202fUmar\u202fAlam**. Let me know if you’d like me to address you differently or if there’s anything specific you’d like help with!'

In [19]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [20]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is M Umar Alam")],"language":"Urdu"})
response.content

'السلام علیکم! میرا نام چیٹ جی پی ٹی ہے۔ آپ کی کیسے مدد کر سکتا ہوں، جناب ایم عمر عالم؟'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [ ]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [22]:
config = {"configurable": {"session_id": "chat4"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am M Umar Alam")],"language":"Urdu"},
    config=config
)
repsonse.content

'اسلام علیکم! آپ سے مل کر خوشی ہوئی، ایم عمر عالم صاحب۔ میں آپ کی کس طرح مدد کر سکتا ہوں؟'

In [23]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Urdu"},
    config=config,
)

In [24]:
response.content

'آپ کا نام ایم عمر عالم ہے۔'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [37]:
import transformers
from transformers import GPT2TokenizerFast

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
print("GPT-2 tokenizer works!")

GPT-2 tokenizer works!


In [39]:

from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=len,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="hi! I'm bob", additional_kwargs={}, response_metadata={}),
 AIMessage(content='hi!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={})]

In [40]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

'You mentioned that you like vanilla ice cream.'

In [41]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'You asked about the sum\u202f2\u202f+\u202f2.'

In [ ]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

In [43]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

'Your name is Bob.'

In [44]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

'You asked for the result of\u202f2\u202f+\u202f2.'